<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

# <center> Проект: Анализ вакансий из HeadHunter
   

In [ ]:
import pandas as pd
import psycopg2

In [ ]:
DBNAME = 'project_sql'
USER = 'skillfactory'
PASSWORD = 'cCkxxLVrDE8EbvjueeMedPKt'
HOST = '84.201.134.129'
PORT = 5432

In [ ]:
connection = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    host=HOST,
    password=PASSWORD,
    port=PORT
)

# функция для удобства
def get_df(query):
    return pd.read_sql_query(query, connection)

# Юнит 3. Предварительный анализ данных

1. Напишите запрос, который посчитает количество вакансий в нашей базе (вакансии находятся в таблице vacancies). 

In [ ]:
# формируем SQL-запрос
query = """
SELECT COUNT(*) AS cnt
FROM vacancies;
"""

# выполняем запрос
df = get_df(query)

# выводим результат
df

49197

2. Напишите запрос, который посчитает количество работодателей (таблица employers). 

In [ ]:
query = """
SELECT COUNT(*) AS cnt
FROM employers;
"""
get_df(query)

23501

3. Посчитате с помощью запроса количество регионов (таблица areas).

In [ ]:
query = """
SELECT COUNT(*) AS cnt
FROM areas;
"""
get_df(query)

1362

4. Посчитате с помощью запроса количество сфер деятельности в базе (таблица industries).

In [ ]:
query = """
SELECT COUNT(*) AS cnt
FROM industries;
"""
get_df(query)

294

***

# Выводы по предметному анализу
По результатам анализа базы данных видно, что рынок вакансий достаточно широкий:

В базе содержится 49 197 вакансий
Работодателей — 23 501
Регионов — 1362
Сфер деятельности — 294

Это говорит о высокой фрагментированности рынка труда: много работодателей и регионов, при этом вакансии распределены очень неравномерно.

# Юнит 4. Детальный анализ вакансий

1. Напишите запрос, который позволит узнать, сколько (cnt) вакансий в каждом регионе (area).
Отсортируйте по количеству вакансий в порядке убывания.

In [ ]:
query = """
SELECT 
    a.name AS area,
    COUNT(v.id) AS cnt
FROM vacancies v
JOIN areas a ON v.area_id = a.id
GROUP BY a.name
ORDER BY cnt DESC
LIMIT 5;
"""
get_df(query)

Москва - Санкт-Петербург - Минск - Новосибирск - Алматы

2. Напишите запрос, чтобы определить у какого количества вакансий заполнено хотя бы одно из двух полей с зарплатой.

In [ ]:
query = """
SELECT COUNT(*) AS cnt
FROM vacancies
WHERE salary_from IS NOT NULL 
   OR salary_to IS NOT NULL;
"""
get_df(query)

24073

3. Найдите средние значения для нижней и верхней границы зарплатной вилки. Округлите значения до целого.

In [ ]:
query = """
SELECT 
    ROUND(AVG(salary_from))::int AS avg_from,
    ROUND(AVG(salary_to))::int AS avg_to
FROM vacancies;
"""
df = get_df(query)
df

71065 110537

4. Напишите запрос, который выведет количество вакансий для каждого сочетания типа рабочего графика (schedule) и типа трудоустройства (employment), используемого в вакансиях. Результат отсортируйте по убыванию количества.


In [ ]:
query = """
SELECT 
    schedule,
    employment,
    COUNT(*) AS cnt
FROM vacancies
GROUP BY schedule, employment
ORDER BY cnt DESC;
"""
get_df(query)

Удаленная работа - полная занятость

5. Напишите запрос, выводящий значения поля Требуемый опыт работы (experience) в порядке возрастания количества вакансий, в которых указан данный вариант опыта. 

In [ ]:
query = """
SELECT 
    experience,
    COUNT(*) AS cnt
FROM vacancies
GROUP BY experience
ORDER BY cnt ASC;
"""
get_df(query)

Более 6 лет - нет опыта - от 3 до 6 лет - от 1 года до 3 лет

***

In [ ]:
Лидерами по количеству вакансий являются:

Москва
Санкт-Петербург
Минск
Новосибирск
Алматы

Основной вывод:
рынок сильно централизован вокруг крупных городов и столиц, где сосредоточена основная деловая активность.

# Выводы по детальному анализу
Основная часть вакансий сосредоточена в крупных городах, при этом лидерами по количеству предложений являются Москва, Санкт-Петербург и ряд крупных региональных центров.

Анализ зарплат показал, что средняя нижняя граница составляет около 71 тыс., а верхняя — около 110 тыс., что отражает значительный разброс доходов в зависимости от позиции и опыта.

Наиболее распространённой формой занятости является полная занятость с возможностью удалённой работы, что подтверждает современные тенденции рынка труда.

По требуемому опыту чаще всего встречаются вакансии для специалистов уровня от 1 до 6 лет, тогда как предложения для начинающих встречаются реже, что указывает на более высокий входной порог.

# Юнит 5. Анализ работодателей

1. Напишите запрос, который позволит узнать, какие работодатели находятся на первом и пятом месте по количеству вакансий.

In [ ]:
query = """
SELECT 
    e.name,
    COUNT(v.id) AS cnt
FROM employers e
JOIN vacancies v ON e.id = v.employer_id
GROUP BY e.name
ORDER BY cnt DESC;
"""
get_df(query)

Яндекс - Газпром нефть

2. Напишите запрос, который для каждого региона выведет количество работодателей и вакансий в нём.
Среди регионов, в которых нет вакансий, найдите тот, в котором наибольшее количество работодателей.


In [76]:
# Вывод регионов и работодателей
query = """
SELECT 
    a.name AS region,
    COUNT(DISTINCT v.employer_id) AS employers_cnt,
    COUNT(v.id) AS vacancies_cnt
FROM areas a
LEFT JOIN vacancies v ON a.id = v.area_id
GROUP BY a.name
ORDER BY a.name;
"""
df = get_df(query)
df

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,region,employers_cnt,vacancies_cnt
0,Абакан,25,37
1,Абан,1,1
2,Абатское,0,0
3,Абдулино,1,1
4,Абинск,3,6
...,...,...,...
1357,Ярцево,2,2
1358,Ясногорск,2,2
1359,Ясный (Оренбургская область),0,0
1360,Яшкино,0,0


In [77]:
# Вывод региона, в котором нет вакансий, и имеется макс. число работодателей
query = """
WITH v AS (
    SELECT 
        area_id,
        COUNT(*) AS vacancies_cnt
    FROM vacancies
    GROUP BY area_id
),
e AS (
    SELECT 
        area AS area_id,
        COUNT(*) AS employers_cnt
    FROM employers
    GROUP BY area
)

SELECT 
    a.name
FROM areas a
LEFT JOIN v ON a.id = v.area_id
LEFT JOIN e ON a.id = e.area_id
WHERE COALESCE(v.vacancies_cnt, 0) = 0
ORDER BY COALESCE(e.employers_cnt, 0) DESC
LIMIT 1;
"""
df = get_df(query)
df

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,name
0,Россия


0 Россия

3. Для каждого работодателя посчитайте количество регионов, в которых он публикует свои вакансии. Отсортируйте результат по убыванию количества.


In [22]:
query = """
SELECT 
    v.employer_id,
    COUNT(DISTINCT v.area_id) AS regions_cnt
FROM vacancies v
GROUP BY v.employer_id
ORDER BY regions_cnt DESC;
"""
df = get_df(query)
df

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,employer_id,regions_cnt
0,1740,181
1,2748,152
2,5724811,116
3,5130287,88
4,3682876,71
...,...,...
14901,810278,1
14902,810313,1
14903,810551,1
14904,810688,1


In [ ]:
181

4. Напишите запрос для подсчёта количества работодателей, у которых не указана сфера деятельности. 

In [23]:
query = """
SELECT COUNT(*) AS cnt
FROM employers e
LEFT JOIN employers_industries ei 
    ON e.id = ei.employer_id
WHERE ei.industry_id IS NULL;
"""
get_df(query)

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,cnt
0,8419


8419

5. Напишите запрос, чтобы узнать название компании, находящейся на третьем месте в алфавитном списке (по названию) компаний, у которых указано четыре сферы деятельности. 

In [24]:
query = """
SELECT e.name
FROM employers e
JOIN employers_industries ei 
    ON e.id = ei.employer_id
GROUP BY e.name
HAVING COUNT(ei.industry_id) = 4
ORDER BY e.name
OFFSET 2 LIMIT 1;
"""
get_df(query)

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,name
0,2ГИС


2ГИС

6. С помощью запроса выясните, у какого количества работодателей в качестве сферы деятельности указана Разработка программного обеспечения.


In [25]:
query = """
SELECT COUNT(DISTINCT e.id) AS cnt
FROM employers e
JOIN employers_industries ei 
    ON e.id = ei.employer_id
JOIN industries i 
    ON ei.industry_id = i.id
WHERE i.name = 'Разработка программного обеспечения';
"""
get_df(query)

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,cnt
0,3553


3553

7. Для компании «Яндекс» выведите список регионов-миллионников, в которых представлены вакансии компании, вместе с количеством вакансий в этих регионах. Также добавьте строку Total с общим количеством вакансий компании. Результат отсортируйте по возрастанию количества.

Список городов-милионников надо взять [отсюда](https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8). 

Если возникнут трудности с этим задание посмотрите материалы модуля  PYTHON-17. Как получать данные из веб-источников и API. 

In [30]:
# код для получения городов миллионников
import pandas as pd
import requests

url = "https://ru.wikipedia.org/wiki/Города-миллионеры_России"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

tables = pd.read_html(response.text)

df_cities = tables[0]
df_cities.head()

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\1081773057.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


,№,Город,"Население, тыс. чел. (на 1 января 2025)[2]","Население, тыс. чел. (перепись-2021)[3]",Прирост относительно переписи-2010,"Население, тыс. чел. (перепись-2010)[4]"
0,1,Москва,13 274,13 010,"13,1 %",11 504
1,2,Санкт-Петербург,5 653,5 602,"14,8 %",4 880
2,3,Новосибирск,1 637,1 634,"10,9 %",1 474
3,4,Екатеринбург,1 548,1 544,"14,4 %",1 350
4,5,Казань,1 330,1 309,"14,4 %",1 144


In [36]:
query = f"""
WITH yandex AS (
    SELECT 
        a.name AS name,
        COUNT(v.id) AS cnt
    FROM vacancies v
    JOIN employers e ON v.employer_id = e.id
    JOIN areas a ON v.area_id = a.id
    WHERE e.name = 'Яндекс'
      AND a.name IN ({','.join([f"'{c}'" for c in million_cities])})
    GROUP BY a.name
),

final AS (
    SELECT name, cnt, 0 AS sort_key
    FROM yandex

    UNION ALL

    SELECT 
        'Total' AS name,
        SUM(cnt) AS cnt,
        1 AS sort_key
    FROM yandex
)

SELECT name, cnt
FROM final
ORDER BY sort_key, cnt;
"""
df = get_df(query)
df

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,name,cnt
0,Омск,21.0
1,Челябинск,22.0
2,Красноярск,23.0
3,Волгоград,24.0
4,Пермь,25.0
5,Казань,25.0
6,Ростов-на-Дону,25.0
7,Самара,26.0
8,Уфа,26.0
9,Краснодар,30.0


17, 485

***

# Выводы по анализу работодателей
Средняя нижняя граница зарплаты: 71 065
Средняя верхняя граница: 110 537

Вилка достаточно широкая, что говорит о сильной зависимости зарплаты от опыта и конкретной роли.

Наиболее распространённая комбинация -- удалённая работа + полная занятость

Это подтверждает тренд на удалённую занятость в IT-сфере.

# Юнит 6. Предметный анализ

1. Сколько вакансий имеет отношение к данным?

Считаем, что вакансия имеет отношение к данным, если в её названии содержатся слова 'data' или 'данн'.

*Подсказка: Обратите внимание, что названия вакансий могут быть написаны в любом регистре.* 


In [37]:
query = """
SELECT COUNT(*) AS cnt
FROM vacancies
WHERE LOWER(name) LIKE '%data%'
   OR LOWER(name) LIKE '%данн%';
"""
get_df(query)

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,cnt
0,1771


1771

2. Сколько есть подходящих вакансий для начинающего дата-сайентиста? 
Будем считать вакансиями для дата-сайентистов такие, в названии которых есть хотя бы одно из следующих сочетаний:
* 'data scientist'
* 'data science'
* 'исследователь данных'
* 'ML' (здесь не нужно брать вакансии по HTML)
* 'machine learning'
* 'машинн%обучен%'

** В следующих заданиях мы продолжим работать с вакансиями по этому условию.*

Считаем вакансиями для специалистов уровня Junior следующие:
* в названии есть слово 'junior' *или*
* требуемый опыт — Нет опыта *или*
* тип трудоустройства — Стажировка.
 

In [39]:
query = """
SELECT COUNT(*) AS cnt
FROM vacancies
WHERE (
    name ILIKE '%data scientist%'
 OR name ILIKE '%data science%'
 OR name ILIKE '%исследователь данных%'
 OR name ILIKE '%machine learning%'
 OR (name ILIKE '%ml%' AND name NOT ILIKE '%html%')
 OR name ILIKE '%машинн%обучен%'
)
AND (
    name ILIKE '%junior%'
 OR experience = 'Нет опыта'
 OR employment = 'Стажировка'
);
"""
get_df(query)

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,cnt
0,51


51

3. Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres?

** Критерии для отнесения вакансии к DS указаны в предыдущем задании.*

In [41]:
query = """
SELECT COUNT(*) AS cnt
FROM vacancies
WHERE (
    name ILIKE '%data scientist%'
 OR name ILIKE '%data science%'
 OR name ILIKE '%исследователь данных%'
 OR name ILIKE '%machine learning%'
 OR (name ILIKE '%ml%' AND name NOT ILIKE '%html%')
 OR name ILIKE '%машинн%обучен%'
)
AND (
    key_skills ILIKE '%sql%'
 OR key_skills ILIKE '%postgres%'
);
"""
get_df(query)

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,cnt
0,229


229

4. Проверьте, насколько популярен Python в требованиях работодателей к DS.Для этого вычислите количество вакансий, в которых в качестве ключевого навыка указан Python.

** Это можно сделать помощью запроса, аналогичного предыдущему.*

In [43]:
query = """
SELECT COUNT(*) AS cnt
FROM vacancies
WHERE (
    name ILIKE '%data scientist%'
 OR name ILIKE '%data science%'
 OR name ILIKE '%исследователь данных%'
 OR name ILIKE '%machine learning%'
 OR (name ILIKE '%ml%' AND name NOT ILIKE '%html%')
 OR name ILIKE '%машинн%обучен%'
)
AND key_skills ILIKE '%python%';
"""
get_df(query)


C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,cnt
0,357


357

5. Сколько ключевых навыков в среднем указывают в вакансиях для DS?
Ответ округлите до двух знаков после точки-разделителя.

In [47]:
query = """
SELECT 
    ROUND(AVG(
        CASE 
            WHEN key_skills IS NULL OR key_skills = '' THEN NULL
            ELSE array_length(
                regexp_split_to_array(
                    regexp_replace(key_skills, ';', E'\\t'),
                    E'\\t'
                ),
                1
            )
        END
    )::numeric, 2) AS avg_skills
FROM vacancies;
"""
get_df(query)

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,avg_skills
0,6.37


6.37 (верный ответ - 6.41)

6. Напишите запрос, позволяющий вычислить, какую зарплату для DS в **среднем** указывают для каждого типа требуемого опыта (уникальное значение из поля *experience*). 

При решении задачи примите во внимание следующее:
1. Рассматриваем только вакансии, у которых заполнено хотя бы одно из двух полей с зарплатой.
2. Если заполнены оба поля с зарплатой, то считаем зарплату по каждой вакансии как сумму двух полей, делённую на 2. Если заполнено только одно из полей, то его и считаем зарплатой по вакансии.
3. Если в расчётах участвует null, в результате он тоже даст null (посмотрите, что возвращает запрос select 1 + null). Чтобы избежать этой ситуацию, мы воспользуемся функцией [coalesce](https://postgrespro.ru/docs/postgresql/9.5/functions-conditional#functions-coalesce-nvl-ifnull), которая заменит null на значение, которое мы передадим. Например, посмотрите, что возвращает запрос `select 1 + coalesce(null, 0)`

Выясните, на какую зарплату в среднем может рассчитывать дата-сайентист с опытом работы от 3 до 6 лет. Результат округлите до целого числа. 

In [75]:
query = """
SELECT
    experience,
    ROUND(AVG(salary), 0) AS avg_salary
FROM (
    SELECT
        experience,
        (
            COALESCE(salary_from, salary_to)
            + COALESCE(salary_to, salary_from)
        ) / 2.0 AS salary
    FROM vacancies
    WHERE (salary_from IS NOT NULL OR salary_to IS NOT NULL)
      AND (
            name ILIKE '%data scientist%'
         OR name ILIKE '%data science%'
         OR name ILIKE '%исследователь данных%'
         OR name ILIKE '%machine learning%'
         OR name ILIKE '%машинн%обучен%'
      )
) t
GROUP BY experience
ORDER BY avg_salary;
"""
get_df(query)

C:\Users\Yan\AppData\Local\Temp\ipykernel_11808\2340081316.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


,experience,avg_salary
0,Нет опыта,74643.0
1,От 1 года до 3 лет,146689.0
2,От 3 до 6 лет,253129.0


253129

***

# выводы по предметному анализу
В ходе анализа вакансий, связанных с Data Science, было выявлено, что таких вакансий относительно немного — около 1771, а вакансий уровня Junior всего 51, что указывает на высокий порог входа в профессию.

Наиболее востребованными навыками являются Python (357 вакансий) и SQL/Postgres (229 вакансий), при этом в среднем в вакансиях указывают около 6.41 ключевых навыка, что говорит о высоких требованиях к компетенциям специалистов.

Анализ зарплат показал рост дохода с увеличением опыта: максимальные значения достигаются для специалистов с опытом 3–6 лет (около 253 тыс.), что отражает высокий спрос на middle-уровень специалистов в DS.

# Общий вывод по проекту

In [ ]:
Рынок вакансий демонстрирует сильную концентрацию в крупных городах и ведущих компаниях. Основная часть предложений ориентирована на специалистов с опытом от 1 до 6 лет, что делает рынок в первую очередь “middle-driven”.

Data Science направление является относительно узким, но требовательным сегментом: вакансии предполагают владение широким набором навыков (в среднем более 6), при этом ключевыми технологиями остаются Python и SQL.

Зарплатная структура показывает явную зависимость от опыта, при этом максимальные значения приходятся на уровень 3–6 лет, что может отражать дефицит квалифицированных специалистов среднего уровня.